# Chapter 14: Algorithm Designs and Techniques

Solve problems by splitting them, remembering smaller answers, making local choices, or undoing choices that fail. This notebook implements the supplied chapter's algorithms, with worked examples, reconstructed solutions, and checks.

Select **JavaScript (Node.js)**. Every code cell is self-contained and can be run independently or rerun without declaration conflicts. No external packages are required. `console.table` displays intermediate tables; assertions silently pass or throw if an answer is incorrect. Run all cells for the full lesson.

| Technique | Main idea | Examples here |
| --- | --- | --- |
| Divide and conquer | Solve smaller instances | Recursive binary search |
| Dynamic programming | Reuse overlapping subproblem answers | Coins, knapsack, LCS, matrix chains, Floyd–Warshall |
| Greedy | Make a locally best choice | Coins, fractional knapsack |
| Backtracking | Choose, explore, undo on failure | Maze, Sudoku |
| Functional style | Express transformations through functions | map, filter, reduce |

Top-down **memoization** caches recursive results. Bottom-up **tabulation** fills a table in dependency order. Both are dynamic programming; the iterative tables below are tabulation. Inputs are small teaching examples using fixed-size numeric arithmetic unless stated otherwise.

## 1. Recursive binary search

The array must already be sorted according to the comparator. Compute the middle index, recurse into the half that may contain the target, and stop on equality or an empty range. No merge step is needed. More precisely this is decrease-and-conquer: only one half is solved.

`binarySearch` returns an index in the supplied sorted array, or −1. `sortAndBinarySearch` explicitly sorts a copy and returns both that copy and its index, preserving the original input. An index in sorted data is generally different from an original input index.

The example finds 7 at index 3 in [1, 3, 5, 7, 9]. On sorted input the search costs O(log n) time and O(log n) recursive stack space. Sorting first adds the sort's time and memory costs. Comparator results are interpreted by sign, so ordinary `(a, b) => a - b` works.

In [1]:
(() => {
const assert = require("node:assert/strict");
function binarySearchRecursive(array, value, low, high, compareFn = (a, b) => a - b) {
  if (low > high) return -1;
  const mid = low + Math.floor((high - low) / 2);
  const order = compareFn(array[mid], value);
  if (order < 0) return binarySearchRecursive(array, value, mid + 1, high, compareFn);
  if (order > 0) return binarySearchRecursive(array, value, low, mid - 1, compareFn);
  return mid;
}
function binarySearch(array, value, compareFn = (a, b) => a - b) {
  return binarySearchRecursive(array, value, 0, array.length - 1, compareFn);
}
function sortAndBinarySearch(array, value, compareFn = (a, b) => a - b) {
  const sorted = array.slice().sort(compareFn);
  return { sorted, index: binarySearch(sorted, value, compareFn) };
}

assert.equal(binarySearch([1, 3, 5, 7, 9], 7), 3);
assert.equal(binarySearch([1, 3, 5], 2), -1);
console.log(sortAndBinarySearch([9, 1, 7, 3, 5], 7));
})();

{ sorted: [ 1, 3, 5, 7, 9 ], index: 3 }


## 2. Minimum coins: tabulation and memoization

Given unlimited copies of each denomination, find the fewest coins making an exact amount. Define best[0] = 0 and best[a] = min(best[a − coin] + 1) over usable coins. Infinity marks unreachable amounts. Store the chosen last coin to reconstruct an answer.

The bottom-up version returns `{ coins, count, table }`; `coins: null` and `count: Infinity` mean no exact solution. Amount zero returns an empty list, which is a successful solution. The memoized version returns a coin list or null and stores counts and choices rather than copying a list for every subproblem.

For [1, 3, 4] and amount 6, both find [3, 3]. For US denominations and 36, an optimal answer contains 25, 10, and 1; order does not affect its validity. With d denominations and amount A, both use O(dA) time and O(A) storage, including choices and the recursive stack in the memoized version. This is pseudopolynomial in A, not polynomial in the number of bits encoding A. Recursion can overflow the JavaScript stack for large amounts; use the iterative version then.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function validateCoins(coins, amount) {
  if (!Number.isSafeInteger(amount) || amount < 0 ||
      coins.some(coin => !Number.isSafeInteger(coin) || coin <= 0)) {
    throw new RangeError('Use positive integer coins and a nonnegative integer amount.');
  }
}
function minCoinChange(coins, amount) {
  validateCoins(coins, amount);
  const best = Array(amount + 1).fill(Infinity), choice = Array(amount + 1).fill(null);
  best[0] = 0;
  for (let a = 1; a <= amount; a++) {
    for (const coin of coins) {
      if (coin <= a && best[a - coin] + 1 < best[a]) {
        best[a] = best[a - coin] + 1;
        choice[a] = coin;
      }
    }
  }
  if (best[amount] === Infinity) return { coins: null, count: Infinity, table: best };
  const result = [];
  for (let a = amount; a > 0; a -= choice[a]) result.push(choice[a]);
  return { coins: result, count: best[amount], table: best };
}
function minCoinChangeMemo(coins, amount) {
  validateCoins(coins, amount);
  const cache = new Map([[0, 0]]), choice = new Map();
  function solve(value) {
    if (cache.has(value)) return cache.get(value);
    let best = Infinity;
    for (const coin of coins) if (coin <= value) {
      const count = solve(value - coin) + 1;
      if (count < best) { best = count; choice.set(value, coin); }
    }
    cache.set(value, best);
    return best;
  }
  if (solve(amount) === Infinity) return null;
  const result = [];
  for (let a = amount; a > 0; a -= choice.get(a)) result.push(choice.get(a));
  return result;
}

const result = minCoinChange([1, 3, 4], 6);
console.table(result.table.map((count, amount) => ({ amount, count })));
assert.deepEqual(result.coins, [3, 3]);
assert.deepEqual(minCoinChangeMemo([1, 3, 4], 6), [3, 3]);
assert.equal(minCoinChange([1, 5, 10, 25], 36).count, 3);
assert.equal(minCoinChange([4], 6).coins, null);
console.log('Minimum change:', result.coins);
})();

## 3. Count coin combinations

Counting ways is a different question from minimizing coins. Process each distinct denomination in the outer loop; update amounts upward so that a coin can be reused. ways[0] = 1 represents the empty combination. Denomination-first ordering counts [1, 3] and [3, 1] as the same combination.

For [1, 2, 5] and amount 5 there are four combinations: five ones; three ones and a two; one one and two twos; one five. The return value is BigInt so large counts remain exact. There are O(dA) additions and O(A) stored counts; BigInt addition cost depends on count size.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function validateCoins(coins, amount) {
  if (!Number.isSafeInteger(amount) || amount < 0 ||
      coins.some(coin => !Number.isSafeInteger(coin) || coin <= 0)) {
    throw new RangeError('Use positive integer coins and a nonnegative integer amount.');
  }
}
function countCoinChange(coins, amount) {
  validateCoins(coins, amount);
  const ways = Array(amount + 1).fill(0n);
  ways[0] = 1n;
  for (const coin of new Set(coins)) {
    for (let a = coin; a <= amount; a++) ways[a] += ways[a - coin];
  }
  return ways[amount];
}

assert.equal(countCoinChange([1, 2, 5], 5), 4n);
assert.equal(countCoinChange([1, 1, 2], 3), 2n);
console.log('Combinations for 5:', String(countCoinChange([1, 2, 5], 5)));
})();

## 4. 0–1 knapsack and selected items

Each item can be selected once, whole. dp[i][w] means the best value using the first i items within capacity w. Either skip item i or take it and add the best answer from the **previous row** at the remaining capacity. Using the current row instead would allow repeated copies.

Reconstruct backward: when the value differs from the previous row, select that item and subtract its **weight** from the remaining capacity. On ties this version skips the item. Returned indices are zero-based and sorted in input order. The input arrays are preserved.

For weights [2, 3, 4], values [3, 4, 5], and capacity 5, select indices [0, 1]: total weight 5, total value 7. At capacity 6 select [0, 2] for value 8. Time and table space are O(nW), pseudopolynomial in integer capacity W. A memoized recursive variant is also included; reconstruction is provided by the tabulated version.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function validateItems(capacity, weights, values, integer = false) {
  if (!Number.isFinite(capacity) || capacity < 0 || (integer && !Number.isSafeInteger(capacity)) ||
      weights.length !== values.length || weights.some(w => !Number.isFinite(w) || w <= 0 || (integer && !Number.isSafeInteger(w))) ||
      values.some(v => !Number.isFinite(v) || v < 0)) {
    throw new RangeError('Use matching item arrays, positive weights, nonnegative values/capacity, and integer DP weights/capacity.');
  }
}
function knapSack(capacity, weights, values) {
  validateItems(capacity, weights, values, true);
  const n = values.length;
  const table = Array.from({ length: n + 1 }, () => Array(capacity + 1).fill(0));
  for (let i = 1; i <= n; i++) for (let w = 0; w <= capacity; w++) {
    table[i][w] = table[i - 1][w];
    if (weights[i - 1] <= w) table[i][w] = Math.max(table[i][w],
      values[i - 1] + table[i - 1][w - weights[i - 1]]);
  }
  const indices = [];
  let w = capacity;
  for (let i = n; i > 0; i--) if (table[i][w] !== table[i - 1][w]) {
    indices.push(i - 1);
    w -= weights[i - 1];
  }
  indices.reverse();
  return { value: table[n][capacity], indices, weight: capacity - w, table };
}
function knapSackMemo(capacity, weights, values) {
  validateItems(capacity, weights, values, true);
  const cache = new Map();
  function solve(i, remaining) {
    if (i === values.length) return 0;
    const key = `${i},${remaining}`;
    if (cache.has(key)) return cache.get(key);
    let best = solve(i + 1, remaining);
    if (weights[i] <= remaining) best = Math.max(best, values[i] + solve(i + 1, remaining - weights[i]));
    cache.set(key, best);
    return best;
  }
  return solve(0, capacity);
}

const result = knapSack(5, [2, 3, 4], [3, 4, 5]);
console.table(result.table);
console.log('Selected indices:', result.indices, 'weight:', result.weight, 'value:', result.value);
assert.equal(result.value, 7);
assert.deepEqual(result.indices, [0, 1]);
assert.equal(result.weight, 5);
assert.equal(knapSackMemo(5, [2, 3, 4], [3, 4, 5]), 7);
assert.deepEqual(knapSack(6, [2, 3, 4], [3, 4, 5]).indices, [0, 2]);
})();

## 5. Longest common subsequence

A subsequence preserves relative order but can skip characters. It need not be a contiguous substring. Let dp[i][j] be the LCS length of the first i characters of X and first j of Y. Equal final characters extend the diagonal by one; otherwise take the larger answer above or to the left.

Walk backward to recover a sequence: take an equal character and move diagonally, or move toward the larger neighboring value. Multiple LCS answers may be valid; ties here move upward. For acbaed and abcadf, the recovered answer is acad with length 4. The cell includes a recursive memoized length-only alternative.

For lengths m and n, time and table storage are O(mn), with up to O(m + n) reconstruction steps. The recursive alternative adds up to O(m + n) stack depth. String positions use UTF-16 code units.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function lcs(wordX, wordY) {
  const m = wordX.length, n = wordY.length;
  const table = Array.from({ length: m + 1 }, () => Array(n + 1).fill(0));
  for (let i = 1; i <= m; i++) for (let j = 1; j <= n; j++) {
    table[i][j] = wordX[i - 1] === wordY[j - 1] ? table[i - 1][j - 1] + 1 : Math.max(table[i - 1][j], table[i][j - 1]);
  }
  const characters = [];
  let i = m, j = n;
  while (i > 0 && j > 0) {
    if (wordX[i - 1] === wordY[j - 1]) { characters.push(wordX[i - 1]); i--; j--; }
    else if (table[i - 1][j] >= table[i][j - 1]) i--;
    else j--;
  }
  return { length: table[m][n], sequence: characters.reverse().join(''), table };
}
function lcsMemo(x, y) {
  const cache = new Map();
  function solve(i, j) {
    if (i === 0 || j === 0) return 0;
    const key = `${i},${j}`;
    if (!cache.has(key)) cache.set(key, x[i - 1] === y[j - 1] ?
      solve(i - 1, j - 1) + 1 : Math.max(solve(i - 1, j), solve(i, j - 1)));
    return cache.get(key);
  }
  return solve(x.length, y.length);
}

const result = lcs('acbaed', 'abcadf');
console.table(result.table);
console.log('LCS:', result.sequence, 'length:', result.length);
assert.equal(result.length, 4);
assert.equal(result.sequence, 'acad');
assert.equal(lcsMemo('acbaed', 'abcadf'), 4);
})();

## 6. Matrix-chain multiplication order

Multiplying an r×s matrix by an s×t matrix produces r×t and requires rst scalar multiplications with the usual algorithm. Associativity permits different parenthesizations while preserving matrix order; it does not permit arbitrary reordering.

Dimensions p = [10, 100, 5, 50, 1] describe four compatible matrices. dp[i][j] tries every split k: left cost + right cost + p[i]p[k + 1]p[j + 1]. A single matrix has zero multiplication cost. Store the best split to reconstruct the parenthesization.

The answer is (A1(A2(A3A4))) with cost 1750: 250 for CD, 500 for B(CD), and 1000 for A(B(CD)). No actual matrices are multiplied. For N matrices, time is O(N³), table space O(N²). Dimensions and resulting counts must fit exact safe-integer Number arithmetic; this implementation rejects an unsafe candidate cost.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function matrixChainOrder(p) {
  if (p.length < 2 || p.some(x => !Number.isSafeInteger(x) || x <= 0)) throw new RangeError('Supply at least two positive integer dimensions.');
  const n = p.length - 1;
  const table = Array.from({ length: n }, () => Array(n).fill(0));
  const split = Array.from({ length: n }, () => Array(n).fill(null));
  for (let length = 2; length <= n; length++) for (let i = 0; i + length <= n; i++) {
    const j = i + length - 1;
    table[i][j] = Infinity;
    for (let k = i; k < j; k++) {
      const cost = table[i][k] + table[k + 1][j] + p[i] * p[k + 1] * p[j + 1];
      if (!Number.isSafeInteger(cost)) throw new RangeError('Cost exceeds exact Number arithmetic.');
      if (cost < table[i][j]) { table[i][j] = cost; split[i][j] = k; }
    }
  }
  function order(i, j) {
    if (i === j) return `A${i + 1}`;
    return `(${order(i, split[i][j])}${order(split[i][j] + 1, j)})`;
  }
  return { cost: table[0][n - 1], order: order(0, n - 1), table, split };
}

const result = matrixChainOrder([10, 100, 5, 50, 1]);
console.table(result.table);
console.log('Best order:', result.order, 'scalar multiplications:', result.cost);
assert.equal(result.cost, 1750);
assert.equal(result.order, '(A1(A2(A3A4)))');
})();

## 7. All-pairs shortest paths: Floyd–Warshall

This DP example revisits Chapter 12. Start with edge weights, Infinity for absent edges, and zero diagonal entries. For each allowed intermediate vertex k, improve every pair i, j using distance[i][k] + distance[k][j]. The k loop must be outermost: after it finishes, paths may use vertices through k as intermediates.

The function copies the adjacency matrix, returns distances, and detects a negative cycle via a negative final diagonal. If a negative cycle exists, affected pairs do not have finite shortest paths; do not interpret those table entries as valid distances. Negative edges are allowed. Time is O(V³) and extra space O(V²).

In [ ]:
(() => {
const assert = require("node:assert/strict");
function floydWarshall(graph) {
  const n = graph.length;
  if (graph.some(row => row.length !== n || row.some(x => typeof x !== 'number' || (!Number.isFinite(x) && x !== Infinity)))) {
    throw new RangeError('Use a square matrix of finite weights or Infinity.');
  }
  const distances = graph.map(row => row.slice());
  for (let i = 0; i < n; i++) distances[i][i] = Math.min(0, distances[i][i]);
  for (let k = 0; k < n; k++) for (let i = 0; i < n; i++) for (let j = 0; j < n; j++) {
    distances[i][j] = Math.min(distances[i][j], distances[i][k] + distances[k][j]);
  }
  return { distances, negativeCycle: distances.some((row, i) => row[i] < 0) };
}

const result = floydWarshall([[0, 3, 10], [Infinity, 0, 2], [Infinity, Infinity, 0]]);
console.table(result.distances);
assert.equal(result.distances[0][2], 5);
assert.equal(result.negativeCycle, false);
})();

## 8. Greedy coin change: fast choices can fail

Sort denominations in descending order, take as many of the largest coin as possible, and continue. This locally best choice is optimal for some denomination systems, but needs a proof for a particular system.

For US coins and 36 the result is [25, 10, 1]. For [1, 3, 4] and 6 it gives [4, 1, 1], while DP finds [3, 3]. More seriously, for [3, 4] and 6, greedy takes 4 and gets stuck although 3 + 3 works. Return the remainder explicitly so partial change is never mistaken for an exact solution.

With d input denominations and C emitted coins, the comparison-sort model gives O(d log d + C) time, and O(d + C) storage for the sorted copy and result. Inputs are preserved.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function validateCoins(coins, amount) {
  if (!Number.isSafeInteger(amount) || amount < 0 ||
      coins.some(coin => !Number.isSafeInteger(coin) || coin <= 0)) {
    throw new RangeError('Use positive integer coins and a nonnegative integer amount.');
  }
}
function greedyCoinChange(coins, amount) {
  validateCoins(coins, amount);
  const sorted = [...new Set(coins)].sort((a, b) => b - a), change = [];
  let remainder = amount;
  for (const coin of sorted) while (coin <= remainder) {
    change.push(coin);
    remainder -= coin;
  }
  return { coins: change, remainder, exact: remainder === 0 };
}

assert.deepEqual(greedyCoinChange([1, 5, 10, 25], 36).coins, [25, 10, 1]);
assert.deepEqual(greedyCoinChange([1, 3, 4], 6).coins, [4, 1, 1]);
assert.deepEqual(greedyCoinChange([3, 4], 6), { coins: [4], remainder: 2, exact: false });
console.log('Greedy for 6:', greedyCoinChange([1, 3, 4], 6));
console.log('Greedy gets stuck:', greedyCoinChange([3, 4], 6));
})();

## 9. Fractional knapsack

When items are divisible, sort by value/weight in descending order, take full items while possible, then take the fraction that fills the remaining capacity. Ordering by density is essential; original input order alone cannot guarantee optimality.

For weights [2, 3, 4], values [3, 4, 5], capacity 6, take all of the first two items and one quarter of the third: weight 6, value 8.25. Load increases by fraction × weight, not the whole item's weight. Returned records retain original item indices even after sorting.

This greedy choice is optimal for divisible, positive-weight, nonnegative-value items: replacing a lower-density portion with available higher-density weight cannot decrease value. That argument fails for indivisible 0–1 items. Time is O(n log n) under comparison sorting and storage O(n). Floating-point results may need tolerance checks.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function validateItems(capacity, weights, values, integer = false) {
  if (!Number.isFinite(capacity) || capacity < 0 || (integer && !Number.isSafeInteger(capacity)) ||
      weights.length !== values.length || weights.some(w => !Number.isFinite(w) || w <= 0 || (integer && !Number.isSafeInteger(w))) ||
      values.some(v => !Number.isFinite(v) || v < 0)) {
    throw new RangeError('Use matching item arrays, positive weights, nonnegative values/capacity, and integer DP weights/capacity.');
  }
}
function fractionalKnapsack(capacity, weights, values) {
  validateItems(capacity, weights, values);
  const items = weights.map((weight, index) => ({ index, weight, value: values[index], ratio: values[index] / weight }))
    .sort((a, b) => b.ratio - a.ratio || a.index - b.index);
  let load = 0, value = 0;
  const selected = [];
  for (const item of items) {
    if (load >= capacity) break;
    const fraction = Math.min(1, (capacity - load) / item.weight);
    load += fraction * item.weight;
    value += fraction * item.value;
    selected.push({ index: item.index, fraction });
  }
  return { value, load, selected };
}

const result = fractionalKnapsack(6, [2, 3, 4], [3, 4, 5]);
console.table(result.selected);
assert.equal(result.value, 8.25);
assert.equal(result.load, 6);
assert.deepEqual(result.selected.map(item => item.fraction), [1, 1, 0.25]);
// Deliberately unsorted densities: the more valuable second item must come first.
assert.equal(fractionalKnapsack(1, [1, 1], [1, 100]).value, 100);
console.log('Fractional value:', result.value);
})();

## 10. Rat in a maze: choose, explore, undo

A 1 is open and a 0 is blocked. Starting at the top-left, try down and then right to reach the bottom-right. Mark the current cell in a solution matrix, explore recursively, and clear its mark when that branch fails. Check bounds and openness **before** accepting the destination.

This matches the supplied implementation's down/right movement; it does not explore up or left. It supports rectangular binary mazes and returns a path matrix or null. The input remains unchanged. Null means no path under these movement rules, rather than an error string requiring special parsing.

A failed-cell set remembers positions from which the destination cannot be reached, avoiding repeated failed suffix searches. This combines backtracking with memoization: O(RC) time and O(RC) total storage, including the returned matrix, for R rows and C columns. Stack depth is O(R + C). Without remembering failed cells, repeated paths can cause exponential work.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function ratInAMaze(maze) {
  if (!maze.length) return null;
  const rows = maze.length, cols = maze[0].length;
  if (!cols || maze.some(row => row.length !== cols || row.some(x => x !== 0 && x !== 1))) {
    throw new RangeError('Use a nonempty rectangular binary maze.');
  }
  const solution = maze.map(row => row.map(() => 0)), failed = new Set();
  function findPath(row, col) {
    if (row >= rows || col >= cols || maze[row][col] === 0 || failed.has(`${row},${col}`)) return false;
    solution[row][col] = 1;
    if (row === rows - 1 && col === cols - 1) return true;
    if (findPath(row + 1, col) || findPath(row, col + 1)) return true;
    solution[row][col] = 0;
    failed.add(`${row},${col}`);
    return false;
  }
  return findPath(0, 0) ? solution : null;
}

const maze = [[1,0,0,0], [1,1,1,1], [0,0,1,0], [0,1,1,1]];
const result = ratInAMaze(maze);
console.table(result);
assert.deepEqual(result, [[1,0,0,0], [1,1,1,0], [0,0,1,0], [0,0,1,1]]);
assert.equal(ratInAMaze([[1,1], [1,0]]), null);
})();

## 11. Sudoku solver

Use zero for an empty cell in a 9×9 grid. Find the first blank, try digits 1 through 9, reject digits already in its row, column, or 3×3 box, and recurse. If the choice leads to failure, reset the cell to zero and try the next digit. No blanks means success only after the initial clues have been validated.

The solver copies the puzzle, preserving the clues and input. It returns a solved grid or null for a contradictory or unsatisfiable puzzle, and rejects invalid shape/digit ranges with an exception. It returns the first solution; it does not establish uniqueness.

For E empty cells, O(9ᴱ) is a loose search-tree bound with fixed-size Sudoku checks. Stack depth is O(E) and the copied board uses 81 cells. Backtracking can still be slow on difficult puzzles; these examples deliberately use a small standard puzzle. The final checks validate every row, column, box, and original clue.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function sudokuSolver(matrix) {
  if (matrix.length !== 9 || matrix.some(row => row.length !== 9 || row.some(x => !Number.isInteger(x) || x < 0 || x > 9))) {
    throw new RangeError('Use a 9 by 9 grid with digits 0 through 9.');
  }
  const board = matrix.map(row => row.slice());
  function isSafe(row, col, digit) {
    for (let i = 0; i < 9; i++) if (board[row][i] === digit || board[i][col] === digit) return false;
    const startRow = row - row % 3, startCol = col - col % 3;
    for (let r = startRow; r < startRow + 3; r++) for (let c = startCol; c < startCol + 3; c++) {
      if (board[r][c] === digit) return false;
    }
    return true;
  }
  for (let r = 0; r < 9; r++) for (let c = 0; c < 9; c++) if (board[r][c]) {
    const digit = board[r][c];
    board[r][c] = 0;
    const valid = isSafe(r, c, digit);
    board[r][c] = digit;
    if (!valid) return null;
  }
  function solve() {
    let row = -1, col = -1;
    outer: for (let r = 0; r < 9; r++) for (let c = 0; c < 9; c++) if (board[r][c] === 0) {
      row = r; col = c; break outer;
    }
    if (row === -1) return true;
    for (let digit = 1; digit <= 9; digit++) if (isSafe(row, col, digit)) {
      board[row][col] = digit;
      if (solve()) return true;
      board[row][col] = 0;
    }
    return false;
  }
  return solve() ? board : null;
}

const puzzle = [
 [5,3,0,0,7,0,0,0,0], [6,0,0,1,9,5,0,0,0], [0,9,8,0,0,0,0,6,0],
 [8,0,0,0,6,0,0,0,3], [4,0,0,8,0,3,0,0,1], [7,0,0,0,2,0,0,0,6],
 [0,6,0,0,0,0,2,8,0], [0,0,0,4,1,9,0,0,5], [0,0,0,0,8,0,0,7,9]
];
const before = JSON.stringify(puzzle);
const solution = sudokuSolver(puzzle);
assert.ok(solution);
console.table(solution);
assert.equal(JSON.stringify(puzzle), before);
const checkUnit = unit => assert.deepEqual(unit.slice().sort((a,b) => a-b), [1,2,3,4,5,6,7,8,9]);
for (let i = 0; i < 9; i++) {
  checkUnit(solution[i]);
  checkUnit(solution.map(row => row[i]));
}
for (let r = 0; r < 9; r += 3) for (let c = 0; c < 9; c += 3) {
  checkUnit(solution.slice(r, r + 3).flatMap(row => row.slice(c, c + 3)));
}
for (let r = 0; r < 9; r++) for (let c = 0; c < 9; c++) if (puzzle[r][c]) assert.equal(solution[r][c], puzzle[r][c]);
})();

## 12. Functional JavaScript: transformations and callbacks

A higher-order function accepts or returns functions. `forEach` applies an action; `map` builds transformed values; `filter` keeps matching values; `reduce` accumulates a result. Pure functions return the same result for the same input and do not change external state. Logging is a side effect, so a callback-based printer is not pure merely because it uses functions. Evaluation order still matters when effects are present.

The examples print each item, find a minimum, extract day values, keep nonnegative numbers (including zero), sum values, and concatenate arrays. Explicit reduce initializers handle empty collections: sum returns 0; minimum returns undefined by this notebook's convention.

`Math.min(...array)` is convenient for small arrays but can exceed function argument limits on large inputs. A reduce-based minimum avoids spreading. map/filter/sum/minimum take O(n) time here; map/filter allocate O(n) output, while reduction to a number uses O(1) extra space. Repeated `concat` in a reducer can repeatedly copy the growing prefix; `flat(1)` expresses the one-level flattening directly. Copies are shallow, so nested objects can remain shared.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function findMinArray(array) {
  if (array.length === 0) return undefined;
  let minimum = array[0];
  for (let i = 1; i < array.length; i++) if (array[i] < minimum) minimum = array[i];
  return minimum;
}
const minimum = array => array.length ? array.reduce((best, value) => Math.min(best, value), Infinity) : undefined;
const nonnegativeNumbers = array => array.filter(value => value >= 0);
const sum = array => array.reduce((total, value) => total + value, 0);
const mergeArraysConcat = arrays => arrays.reduce((merged, array) => merged.concat(array), []);
const mergeArrays = (...arrays) => arrays.flat(1);
const forEach = (array, action) => array.forEach(action);
const logItem = item => console.log(item);

forEach([1, 2, 3, 4, 5], logItem);
const days = [{ name: 'Monday', value: 1 }, { name: 'Tuesday', value: 2 }, { name: 'Wednesday', value: 7 }];
// Values are the supplied chapter's sample data, not weekday numbering.
assert.deepEqual(days.map(day => day.value), [1, 2, 7]);
assert.equal(findMinArray([8, 6, 4, 5, 9]), 4);
assert.equal(minimum([8, 6, 4, 5, 9]), 4);
assert.deepEqual(nonnegativeNumbers([-1, 0, 1, 2, -2]), [0, 1, 2]);
assert.equal(sum([1, 2, 3, 4, 5]), 15);
assert.deepEqual(mergeArrays([1,2,3], [4,5], [6]), [1,2,3,4,5,6]);
assert.deepEqual(mergeArraysConcat([[1,2,3], [4,5], [6]]), [1,2,3,4,5,6]);
console.log('Minimum:', minimum([8,6,4,5,9]), 'sum:', sum([1,2,3,4,5]));
})();

## 13. Checks and small experiments

These checks cover boundary cases, reconstruction validity, input preservation, comparator order, unreachable amounts, greedy counterexamples, blocked maze endpoints, invalid Sudoku clues, and negative cycles. Small exhaustive or independent reference calculations check optimization results rather than simply repeating a recurrence.

Try changing coin denominations and compare greedy with DP. Try permuting fractional-knapsack items: the optimum should stay the same because density sorting is internal. Block the maze's destination and predict null. For LCS, check that the reconstructed answer is a subsequence of both inputs, even when multiple optimal answers exist.

In [ ]:
(() => {
const assert = require("node:assert/strict");
function validateCoins(coins, amount) {
  if (!Number.isSafeInteger(amount) || amount < 0 ||
      coins.some(coin => !Number.isSafeInteger(coin) || coin <= 0)) {
    throw new RangeError('Use positive integer coins and a nonnegative integer amount.');
  }
}

function validateItems(capacity, weights, values, integer = false) {
  if (!Number.isFinite(capacity) || capacity < 0 || (integer && !Number.isSafeInteger(capacity)) ||
      weights.length !== values.length || weights.some(w => !Number.isFinite(w) || w <= 0 || (integer && !Number.isSafeInteger(w))) ||
      values.some(v => !Number.isFinite(v) || v < 0)) {
    throw new RangeError('Use matching item arrays, positive weights, nonnegative values/capacity, and integer DP weights/capacity.');
  }
}

function binarySearchRecursive(array, value, low, high, compareFn = (a, b) => a - b) {
  if (low > high) return -1;
  const mid = low + Math.floor((high - low) / 2);
  const order = compareFn(array[mid], value);
  if (order < 0) return binarySearchRecursive(array, value, mid + 1, high, compareFn);
  if (order > 0) return binarySearchRecursive(array, value, low, mid - 1, compareFn);
  return mid;
}
function binarySearch(array, value, compareFn = (a, b) => a - b) {
  return binarySearchRecursive(array, value, 0, array.length - 1, compareFn);
}
function sortAndBinarySearch(array, value, compareFn = (a, b) => a - b) {
  const sorted = array.slice().sort(compareFn);
  return { sorted, index: binarySearch(sorted, value, compareFn) };
}

function minCoinChange(coins, amount) {
  validateCoins(coins, amount);
  const best = Array(amount + 1).fill(Infinity), choice = Array(amount + 1).fill(null);
  best[0] = 0;
  for (let a = 1; a <= amount; a++) {
    for (const coin of coins) {
      if (coin <= a && best[a - coin] + 1 < best[a]) {
        best[a] = best[a - coin] + 1;
        choice[a] = coin;
      }
    }
  }
  if (best[amount] === Infinity) return { coins: null, count: Infinity, table: best };
  const result = [];
  for (let a = amount; a > 0; a -= choice[a]) result.push(choice[a]);
  return { coins: result, count: best[amount], table: best };
}
function minCoinChangeMemo(coins, amount) {
  validateCoins(coins, amount);
  const cache = new Map([[0, 0]]), choice = new Map();
  function solve(value) {
    if (cache.has(value)) return cache.get(value);
    let best = Infinity;
    for (const coin of coins) if (coin <= value) {
      const count = solve(value - coin) + 1;
      if (count < best) { best = count; choice.set(value, coin); }
    }
    cache.set(value, best);
    return best;
  }
  if (solve(amount) === Infinity) return null;
  const result = [];
  for (let a = amount; a > 0; a -= choice.get(a)) result.push(choice.get(a));
  return result;
}

function countCoinChange(coins, amount) {
  validateCoins(coins, amount);
  const ways = Array(amount + 1).fill(0n);
  ways[0] = 1n;
  for (const coin of new Set(coins)) {
    for (let a = coin; a <= amount; a++) ways[a] += ways[a - coin];
  }
  return ways[amount];
}

function knapSack(capacity, weights, values) {
  validateItems(capacity, weights, values, true);
  const n = values.length;
  const table = Array.from({ length: n + 1 }, () => Array(capacity + 1).fill(0));
  for (let i = 1; i <= n; i++) for (let w = 0; w <= capacity; w++) {
    table[i][w] = table[i - 1][w];
    if (weights[i - 1] <= w) table[i][w] = Math.max(table[i][w],
      values[i - 1] + table[i - 1][w - weights[i - 1]]);
  }
  const indices = [];
  let w = capacity;
  for (let i = n; i > 0; i--) if (table[i][w] !== table[i - 1][w]) {
    indices.push(i - 1);
    w -= weights[i - 1];
  }
  indices.reverse();
  return { value: table[n][capacity], indices, weight: capacity - w, table };
}
function knapSackMemo(capacity, weights, values) {
  validateItems(capacity, weights, values, true);
  const cache = new Map();
  function solve(i, remaining) {
    if (i === values.length) return 0;
    const key = `${i},${remaining}`;
    if (cache.has(key)) return cache.get(key);
    let best = solve(i + 1, remaining);
    if (weights[i] <= remaining) best = Math.max(best, values[i] + solve(i + 1, remaining - weights[i]));
    cache.set(key, best);
    return best;
  }
  return solve(0, capacity);
}

function lcs(wordX, wordY) {
  const m = wordX.length, n = wordY.length;
  const table = Array.from({ length: m + 1 }, () => Array(n + 1).fill(0));
  for (let i = 1; i <= m; i++) for (let j = 1; j <= n; j++) {
    table[i][j] = wordX[i - 1] === wordY[j - 1] ? table[i - 1][j - 1] + 1 : Math.max(table[i - 1][j], table[i][j - 1]);
  }
  const characters = [];
  let i = m, j = n;
  while (i > 0 && j > 0) {
    if (wordX[i - 1] === wordY[j - 1]) { characters.push(wordX[i - 1]); i--; j--; }
    else if (table[i - 1][j] >= table[i][j - 1]) i--;
    else j--;
  }
  return { length: table[m][n], sequence: characters.reverse().join(''), table };
}
function lcsMemo(x, y) {
  const cache = new Map();
  function solve(i, j) {
    if (i === 0 || j === 0) return 0;
    const key = `${i},${j}`;
    if (!cache.has(key)) cache.set(key, x[i - 1] === y[j - 1] ?
      solve(i - 1, j - 1) + 1 : Math.max(solve(i - 1, j), solve(i, j - 1)));
    return cache.get(key);
  }
  return solve(x.length, y.length);
}

function matrixChainOrder(p) {
  if (p.length < 2 || p.some(x => !Number.isSafeInteger(x) || x <= 0)) throw new RangeError('Supply at least two positive integer dimensions.');
  const n = p.length - 1;
  const table = Array.from({ length: n }, () => Array(n).fill(0));
  const split = Array.from({ length: n }, () => Array(n).fill(null));
  for (let length = 2; length <= n; length++) for (let i = 0; i + length <= n; i++) {
    const j = i + length - 1;
    table[i][j] = Infinity;
    for (let k = i; k < j; k++) {
      const cost = table[i][k] + table[k + 1][j] + p[i] * p[k + 1] * p[j + 1];
      if (!Number.isSafeInteger(cost)) throw new RangeError('Cost exceeds exact Number arithmetic.');
      if (cost < table[i][j]) { table[i][j] = cost; split[i][j] = k; }
    }
  }
  function order(i, j) {
    if (i === j) return `A${i + 1}`;
    return `(${order(i, split[i][j])}${order(split[i][j] + 1, j)})`;
  }
  return { cost: table[0][n - 1], order: order(0, n - 1), table, split };
}

function floydWarshall(graph) {
  const n = graph.length;
  if (graph.some(row => row.length !== n || row.some(x => typeof x !== 'number' || (!Number.isFinite(x) && x !== Infinity)))) {
    throw new RangeError('Use a square matrix of finite weights or Infinity.');
  }
  const distances = graph.map(row => row.slice());
  for (let i = 0; i < n; i++) distances[i][i] = Math.min(0, distances[i][i]);
  for (let k = 0; k < n; k++) for (let i = 0; i < n; i++) for (let j = 0; j < n; j++) {
    distances[i][j] = Math.min(distances[i][j], distances[i][k] + distances[k][j]);
  }
  return { distances, negativeCycle: distances.some((row, i) => row[i] < 0) };
}

function greedyCoinChange(coins, amount) {
  validateCoins(coins, amount);
  const sorted = [...new Set(coins)].sort((a, b) => b - a), change = [];
  let remainder = amount;
  for (const coin of sorted) while (coin <= remainder) {
    change.push(coin);
    remainder -= coin;
  }
  return { coins: change, remainder, exact: remainder === 0 };
}

function fractionalKnapsack(capacity, weights, values) {
  validateItems(capacity, weights, values);
  const items = weights.map((weight, index) => ({ index, weight, value: values[index], ratio: values[index] / weight }))
    .sort((a, b) => b.ratio - a.ratio || a.index - b.index);
  let load = 0, value = 0;
  const selected = [];
  for (const item of items) {
    if (load >= capacity) break;
    const fraction = Math.min(1, (capacity - load) / item.weight);
    load += fraction * item.weight;
    value += fraction * item.value;
    selected.push({ index: item.index, fraction });
  }
  return { value, load, selected };
}

function ratInAMaze(maze) {
  if (!maze.length) return null;
  const rows = maze.length, cols = maze[0].length;
  if (!cols || maze.some(row => row.length !== cols || row.some(x => x !== 0 && x !== 1))) {
    throw new RangeError('Use a nonempty rectangular binary maze.');
  }
  const solution = maze.map(row => row.map(() => 0)), failed = new Set();
  function findPath(row, col) {
    if (row >= rows || col >= cols || maze[row][col] === 0 || failed.has(`${row},${col}`)) return false;
    solution[row][col] = 1;
    if (row === rows - 1 && col === cols - 1) return true;
    if (findPath(row + 1, col) || findPath(row, col + 1)) return true;
    solution[row][col] = 0;
    failed.add(`${row},${col}`);
    return false;
  }
  return findPath(0, 0) ? solution : null;
}

function sudokuSolver(matrix) {
  if (matrix.length !== 9 || matrix.some(row => row.length !== 9 || row.some(x => !Number.isInteger(x) || x < 0 || x > 9))) {
    throw new RangeError('Use a 9 by 9 grid with digits 0 through 9.');
  }
  const board = matrix.map(row => row.slice());
  function isSafe(row, col, digit) {
    for (let i = 0; i < 9; i++) if (board[row][i] === digit || board[i][col] === digit) return false;
    const startRow = row - row % 3, startCol = col - col % 3;
    for (let r = startRow; r < startRow + 3; r++) for (let c = startCol; c < startCol + 3; c++) {
      if (board[r][c] === digit) return false;
    }
    return true;
  }
  for (let r = 0; r < 9; r++) for (let c = 0; c < 9; c++) if (board[r][c]) {
    const digit = board[r][c];
    board[r][c] = 0;
    const valid = isSafe(r, c, digit);
    board[r][c] = digit;
    if (!valid) return null;
  }
  function solve() {
    let row = -1, col = -1;
    outer: for (let r = 0; r < 9; r++) for (let c = 0; c < 9; c++) if (board[r][c] === 0) {
      row = r; col = c; break outer;
    }
    if (row === -1) return true;
    for (let digit = 1; digit <= 9; digit++) if (isSafe(row, col, digit)) {
      board[row][col] = digit;
      if (solve()) return true;
      board[row][col] = 0;
    }
    return false;
  }
  return solve() ? board : null;
}

function findMinArray(array) {
  if (array.length === 0) return undefined;
  let minimum = array[0];
  for (let i = 1; i < array.length; i++) if (array[i] < minimum) minimum = array[i];
  return minimum;
}
const minimum = array => array.length ? array.reduce((best, value) => Math.min(best, value), Infinity) : undefined;
const nonnegativeNumbers = array => array.filter(value => value >= 0);
const sum = array => array.reduce((total, value) => total + value, 0);
const mergeArraysConcat = arrays => arrays.reduce((merged, array) => merged.concat(array), []);
const mergeArrays = (...arrays) => arrays.flat(1);
const forEach = (array, action) => array.forEach(action);
const logItem = item => console.log(item);
assert.equal(binarySearch([], 1), -1);
assert.equal(binarySearch([9,7,5,3], 5, (a,b) => b-a), 2);
const original = [3,1,2];
assert.equal(sortAndBinarySearch(original, 2).index, 1);
assert.deepEqual(original, [3,1,2]);
for (const coins of [[], [2], [1,3,4], [3,4], [2,5]]) for (let amount = 0; amount <= 15; amount++) {
  // Breadth-first reference: each level represents one additional coin.
  let frontier = [0], visited = new Set([0]), expected = amount === 0 ? 0 : Infinity;
  for (let depth = 1; frontier.length && expected === Infinity; depth++) {
    const next = [];
    for (const value of frontier) for (const coin of coins) {
      const total = value + coin;
      if (total === amount) expected = Math.min(expected, depth);
      if (total <= amount && !visited.has(total)) { visited.add(total); next.push(total); }
    }
    frontier = next;
  }
  const result = minCoinChange(coins, amount), memo = minCoinChangeMemo(coins, amount);
  assert.equal(result.count, expected);
  assert.equal(memo === null ? Infinity : memo.length, expected);
  if (result.coins !== null) assert.equal(sum(result.coins), amount);
}
assert.throws(() => minCoinChange([0,1], 4), RangeError);
assert.throws(() => greedyCoinChange([1], -1), RangeError);
assert.equal(countCoinChange([], 0), 1n);
assert.equal(countCoinChange([], 3), 0n);
const weights = [2,3,4,1], values = [3,4,5,2];
for (let capacity = 0; capacity <= 12; capacity++) {
  let best = 0;
  for (let mask = 0; mask < 2 ** weights.length; mask++) {
    let weight = 0, value = 0;
    for (let i = 0; i < weights.length; i++) if (mask & (1 << i)) { weight += weights[i]; value += values[i]; }
    if (weight <= capacity) best = Math.max(best, value);
  }
  const result = knapSack(capacity, weights, values);
  assert.equal(result.value, best);
  assert.equal(knapSackMemo(capacity, weights, values), best);
  assert.equal(new Set(result.indices).size, result.indices.length);
  assert.equal(sum(result.indices.map(i => weights[i])), result.weight);
  assert.equal(sum(result.indices.map(i => values[i])), result.value);
  assert.ok(result.weight <= capacity);
}
assert.equal(knapSack(5, [], []).value, 0);
const isSubsequence = (candidate, text) => { let i=0; for (const ch of text) if (ch === candidate[i]) i++; return i === candidate.length; };
for (const x of ['', 'ABC', 'AAAA', 'acbaed']) for (const y of ['', 'BAC', 'AA', 'abcadf']) {
  const result = lcs(x,y);
  let best = 0;
  for (let mask = 0; mask < 2 ** x.length; mask++) {
    const candidate = [...x].filter((_,i) => mask & (1 << i)).join('');
    if (isSubsequence(candidate,y)) best = Math.max(best,candidate.length);
  }
  assert.equal(result.length,best);
  assert.equal(lcsMemo(x,y),best);
  assert.ok(isSubsequence(result.sequence,x) && isSubsequence(result.sequence,y));
  assert.equal(result.sequence.length,best);
}
function bruteChain(p, i, j) {
  if (i === j) return 0;
  let best = Infinity;
  for (let k=i;k<j;k++) best=Math.min(best,bruteChain(p,i,k)+bruteChain(p,k+1,j)+p[i]*p[k+1]*p[j+1]);
  return best;
}
for (const p of [[2,3], [2,3,4], [10,100,5,50,1], [30,35,15,5,10,20,25]]) {
  assert.equal(matrixChainOrder(p).cost, bruteChain(p,0,p.length-2));
}
assert.equal(matrixChainOrder([2,3]).order,'A1');
assert.throws(() => matrixChainOrder([2]), RangeError);
const graph = [[0,3,10],[Infinity,0,2],[Infinity,Infinity,0]];
const graphCopy = graph.map(row => row.slice());
assert.equal(floydWarshall(graph).distances[0][2],5);
assert.deepEqual(graph,graphCopy);
assert.equal(floydWarshall([[0,-2],[1,0]]).negativeCycle,true);
assert.equal(floydWarshall([[0,Infinity],[Infinity,0]]).distances[0][1],Infinity);
for (const capacity of [0,1,5,6,20]) {
  const a=fractionalKnapsack(capacity,[2,3,4],[3,4,5]);
  const b=fractionalKnapsack(capacity,[4,2,3],[5,3,4]);
  assert.ok(Math.abs(a.value-b.value)<1e-10);
  assert.ok(a.load<=capacity+1e-10);
}
assert.equal(fractionalKnapsack(5,[],[]).value,0);
assert.throws(() => fractionalKnapsack(3,[0],[1]), RangeError);
assert.equal(ratInAMaze([]),null);
assert.equal(ratInAMaze([[0]]),null);
assert.deepEqual(ratInAMaze([[1]]),[[1]]);
assert.equal(ratInAMaze([[1,0],[0,1]]),null);
assert.deepEqual(ratInAMaze([[1,1,1]]),[[1,1,1]]);
const maze=[[1,1,1],[1,0,1],[0,0,1]], copy=maze.map(row=>row.slice());
assert.deepEqual(ratInAMaze(maze),[[1,1,1],[0,0,1],[0,0,1]]);
assert.deepEqual(maze,copy);
const solved=Array.from({length:9},(_,r)=>Array.from({length:9},(_,c)=>(r*3+Math.floor(r/3)+c)%9+1));
assert.deepEqual(sudokuSolver(solved),solved);
const invalid=solved.map(row=>row.slice()); invalid[0][0]=invalid[0][1];
assert.equal(sudokuSolver(invalid),null);
// Locally legal clues, but the top-left blank cannot take its only missing row digit.
const impossible=Array.from({length:9},()=>Array(9).fill(0));
impossible[0]=[0,1,2,3,4,5,6,7,8]; impossible[1][0]=9;
assert.equal(sudokuSolver(impossible),null);
assert.throws(()=>sudokuSolver([[0]]),RangeError);
assert.equal(sum([]),0);
assert.equal(minimum([]),undefined);
assert.equal(findMinArray([]),undefined);
assert.deepEqual(mergeArrays(),[]);
assert.deepEqual(mergeArraysConcat([]),[]);
console.log('All Chapter 14 checks passed.');
})();

## Explain without looking

When does caching save repeated work? Why does 0–1 knapsack use the previous row? Why must reconstruction subtract weight? What makes greedy correct for fractional knapsack but unreliable for arbitrary coin systems? What state must be undone during backtracking?

Adapted from the supplied Chapter 14 Algorithm Designs and Techniques text. Corrections include explicit sorted-index semantics, bottom-up tabulation terminology, valid knapsack reconstruction, density sorting for fractional knapsack, compatible matrix dimensions, and blocked-destination and Sudoku-clue checks. Fractional knapsack has an optimal greedy solution; it is not inherently impossible to formulate with other techniques. The N-queens and knight's-tour problems are mentioned in the chapter as further backtracking examples; its worked implementations are the maze and Sudoku.